# MP2: OSS activity analysis — bdowlin2

Part 1: WoC retrieval and source validation. See `bdowlin2_vis.ipynb` for monthly activity and gap analysis.

## Sources and method

Commit membership and historical commit details come **strictly from World of Code**. The Python retrieval code above calls WoC's HTTP API directly, avoiding the Windows build dependency of the optional `python-woc` package. It uses the same documented `p2c` map and `commit` object endpoints. No GitHub commit data is substituted into the WoC summary.

- [WoC remote API documentation](https://worldofcode.org/docs/#/guide_remote)
- [Official Python remote client](https://github.com/ssc-oscar/python-woc/blob/main/woc/remote.py)
- WoC lists the commits associated with a project across its indexed history. This may differ from GitHub's current default branch. Exact author strings are counted; aliases are not merged.
- Author timestamps, in UTC, determine monthly WoC activity. Empty messages are allowed. SHA/project pairs must be unique, and every expected SHA must have details before analysis runs.
- The README describes an older January 2025 snapshot, but some retrieved records are later. We report actual returned date ranges and do not infer present abandonment from a project history endpoint.
- GitHub is used separately for stars, forks, recent themes, status, and supporting evidence. Stars/forks are current retrieval-time values, not reconstructed September 30 counts.
- **Status cutoff: September 30, 2026, 23:59:59 UTC.** Active means the latest default-branch commit returned by GitHub on/before the cutoff is on/after April 1, 2026. This is the user's approved replacement for the README's 2025 cutoff. GitHub uses committer dates for these API filters.
- The ten most recent GitHub commits on/before the cutoff include the latest commit; fewer are retained if fewer are returned. Default-branch history need not equal WoC's project history.


In [1]:
"""Resumable retrieval from WoC's public HTTP API (Python client)."""
from pathlib import Path
import csv,json,time,urllib.request,urllib.parse,urllib.error,sqlite3,os

ROOT=Path(os.environ.get('MP2_ROOT',Path.cwd()))
NETID='bdowlin2'
CACHE=ROOT/'work'/'cache'
CACHE.mkdir(parents=True,exist_ok=True)
FIELDS=['project_wocid','commit_sha1','author','time','commit message']

def get_json(url, attempts=3):
    for attempt in range(attempts):
        try:
            req=urllib.request.Request(url,headers={'User-Agent':'CS445-MP2-research','Accept':'application/json'})
            with urllib.request.urlopen(req,timeout=30) as r: return json.load(r)
        except Exception as e:
            if isinstance(e,urllib.error.HTTPError) and e.code in (400,404,422): raise
            if attempt==attempts-1: raise
            delay=max(5*2**attempt,int(e.headers.get('Retry-After','0')) if isinstance(e,urllib.error.HTTPError) else 0)
            print(f'Retry {attempt+1}: {type(e).__name__}; waiting {delay}s',flush=True)
            time.sleep(delay)

def projects():
    with (ROOT/'net2prj.csv').open(encoding='utf-8-sig',newline='') as f:
        return [r for r in csv.DictReader(f) if r['netID']==NETID]

def get_shas():
    result={}
    for p in projects():
        path=CACHE/(p['WoC']+'_shas.json')
        if path.exists(): shas=json.loads(path.read_text())
        else:
            shas=[]; cursor=0
            while cursor is not None:
                d=get_json('https://worldofcode.org/api/lookup/map/p2c/'+p['WoC']+'?cursor='+str(cursor))
                shas.extend(d['data']); cursor=d.get('nextCursor'); time.sleep(.5)
            shas=sorted(set(shas)); path.write_text(json.dumps(shas))
        result[p['WoC']]=shas
        print(p['WoC'],len(shas),'SHAs',flush=True)
    with (ROOT/(NETID+'_commit_shas.csv')).open('w',encoding='utf-8',newline='') as f:
        w=csv.writer(f); w.writerow(FIELDS[:2]); w.writerows((p,s) for p,ss in result.items() for s in ss)
    return result

def fetch_batch(keys):
    # WoC's endpoint accepts at most 10 keys per request.
    u='https://worldofcode.org/api/lookup/object/commit?'+urllib.parse.urlencode([('q',k) for k in keys])
    return get_json(u)

def retrieve():
    shas=get_shas()
    db=sqlite3.connect(CACHE/'commits.sqlite')
    db.execute('CREATE TABLE IF NOT EXISTS commits (sha TEXT PRIMARY KEY, author TEXT, time INTEGER, message TEXT)')
    done={r[0] for r in db.execute('SELECT sha FROM commits')}
    # Finish smaller projects first, while keeping each SHA unique.
    ordered=dict.fromkeys(s for p in sorted(shas,key=lambda p:len(shas[p])) for s in shas[p])
    missing=[s for s in ordered if s not in done]
    print('Need details:',len(missing),'Cached:',len(done),flush=True)
    failures={}; started=time.monotonic()
    consecutive_failures=0
    for start in range(0,len(missing),10):
        batch=missing[start:start+10]
        try:
            d=fetch_batch(batch)
            failures.update(d.get('errors',{}))
            for sha,c in d['data'].items():
                assert sha in batch and len(c)==5 and len(c[2])==3,(sha,c)
                db.execute('INSERT OR REPLACE INTO commits VALUES (?,?,?,?)',(sha,c[2][0],int(c[2][1]),c[4]))
            consecutive_failures=0
        except Exception as e:
            for sha in batch: failures[sha]=repr(e)
            consecutive_failures+=1
        db.commit()
        if start%100==0 or start+10>=len(missing):
            print(f'Details attempted {min(start+10,len(missing))}/{len(missing)}; cached {db.execute("SELECT count(*) FROM commits").fetchone()[0]}; elapsed {time.monotonic()-started:.0f}s',flush=True)
        if consecutive_failures>=2:
            print('Stopping after two failed batches; all successful details are checkpointed.',flush=True)
            break
        time.sleep(2) # Ten commits per request, sequential requests with waits.
    data={r[0]:r[1:] for r in db.execute('SELECT sha,author,time,message FROM commits')}
    absent={p:[s for s in ss if s not in data] for p,ss in shas.items()}
    (CACHE/'missing_details.json').write_text(json.dumps(absent,indent=2))
    (CACHE/'retrieval_errors.json').write_text(json.dumps(failures,indent=2))
    if any(absent.values()):
        print('INCOMPLETE:',{p:len(ss) for p,ss in absent.items() if ss},flush=True)
        raise RuntimeError('Incomplete retrieval. Rerun to retry missing data; final CSV not replaced.')
    target=ROOT/(NETID+'_project_summary.csv')
    tmp=target.with_suffix('.csv.tmp')
    with tmp.open('w',encoding='utf-8',newline='') as f:
        w=csv.writer(f,delimiter=';'); w.writerow(FIELDS)
        for p,ss in shas.items():
            for s in sorted(ss,key=lambda s:(data[s][1],s)): w.writerow([p,s,*data[s]])
    tmp.replace(target)
    print('COMPLETE:',sum(map(len,shas.values())),'project-commit records',flush=True)


import json,time
from datetime import datetime,timezone

def collect():
    results=[]
    for p in projects():
        repo=p['GH'].split('github.com/')[1].rstrip('/')
        path=CACHE/(p['WoC']+'_github.json')
        if path.exists():
            result=json.loads(path.read_text(encoding='utf-8'))
        else:
            base='https://api.github.com/repos/'+repo
            detail=get_json(base)
            time.sleep(.5)
            recent=get_json(base+'/commits?per_page=10&until=2026-09-30T23%3A59%3A59Z')
            result={'project':p['WoC'],'url':p['GH'],'retrieved_at':datetime.now(timezone.utc).isoformat(),'repository':detail,'recent_commits_asof':recent,'cutoff':'2026-09-30T23:59:59Z'}
            path.write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
            time.sleep(.5)
        print(p['WoC'],result['repository']['stargazers_count'],result['repository']['forks_count'],result['recent_commits_asof'][0]['commit']['committer']['date'] if result['recent_commits_asof'] else 'NO COMMITS',flush=True)
        results.append(result)
    return results


from pathlib import Path
import os
import pandas as pd
from IPython.display import display, Image
ROOT = Path(os.environ.get('MP2_ROOT', Path.cwd()))
NETID = 'bdowlin2'
FIELDS = ['project_wocid','commit_sha1','author','time','commit message']
EXPECTED = {'avehtari_bda_course_aalto': 2272, 'eberharf_cfl': 1138, 'openda-association_openda': 3209, 'openms_openms': 54864, 'vatlab_sos': 8834, 'jayggg_ngstents': 454, 'inqwire_sqir': 1338, 'tee-lab_pyddsde': 233, 'jgcri_gcam-core': 10688, 'windweller_disextract': 559}

def read_submission():
    data = pd.read_csv(ROOT / 'bdowlin2_project_summary.csv', sep=';', keep_default_na=False,
                       dtype={'project_wocid':str,'commit_sha1':str,'author':str,'time':'int64','commit message':str})
    assert list(data.columns) == FIELDS
    assert not data.duplicated(['project_wocid','commit_sha1']).any()
    assert data.author.ne('').all() and data.commit_sha1.str.fullmatch('[0-9a-f]{40}').all()
    assert set(data.project_wocid) <= set(EXPECTED)
    for name, part in data.groupby('project_wocid'):
        assert len(part) == EXPECTED[name], f'Incomplete submitted project: {name}'
    data['date'] = pd.to_datetime(data.time, unit='s', utc=True)
    return data

assigned = pd.DataFrame(projects())
display(assigned)

,netID,WoC,GH
0,bdowlin2,avehtari_bda_course_aalto,https://github.com/avehtari/BDA_course_Aalto
1,bdowlin2,eberharf_cfl,https://github.com/eberharf/cfl
2,bdowlin2,openda-association_openda,https://github.com/OpenDA-Association/OpenDA
3,bdowlin2,openms_openms,https://github.com/OpenMS/OpenMS
4,bdowlin2,vatlab_sos,https://github.com/vatlab/SoS
5,bdowlin2,jayggg_ngstents,https://github.com/jayggg/ngstents
6,bdowlin2,inqwire_sqir,https://github.com/inQWIRE/SQIR
7,bdowlin2,tee-lab_pyddsde,https://github.com/tee-lab/pyddsde
8,bdowlin2,jgcri_gcam-core,https://github.com/JGCRI/gcam-core
9,bdowlin2,windweller_disextract,https://github.com/windweller/DisExtract


## Retrieve and resume

The retrieval functions are included above; no local helper scripts are required. Set `RUN_NETWORK = True` only to start or resume retrieval. Do not start another retriever while the background collector is running. Each sequential request contains ten commits, followed by a two-second wait. Successful records are checkpointed in a locally generated SQLite database.

With `RUN_NETWORK = False`, this notebook reads the submitted CSV and runs without pre-existing local caches. The CSV currently contains seven complete projects, not all ten. Expected counts below come from the saved WoC manifests; exact SHA-set coverage was verified against those manifests before exporting this snapshot.

In [2]:
RUN_NETWORK = False
if RUN_NETWORK:
    collect()
    retrieve()
else:
    print('Offline submission review; no network retrieval started.')

Offline submission review; no network retrieval started.


In [3]:
summary = read_submission()
submitted = summary.groupby('project_wocid').size().to_dict()
coverage = pd.DataFrame([{'Project':name, 'Expected commits':count,
                         'Details in submitted CSV':submitted.get(name,0),
                         'Complete':submitted.get(name,0)==count} for name,count in EXPECTED.items()])
display(coverage)
print('Submitted complete projects:', int(coverage['Complete'].sum()), '/ 10')
print('Other partially retrieved projects are excluded from this CSV snapshot.')

,Project,Expected commits,Details in submitted CSV,Complete
0,avehtari_bda_course_aalto,2272,2272,True
1,eberharf_cfl,1138,1138,True
2,openda-association_openda,3209,0,False
3,openms_openms,54864,0,False
4,vatlab_sos,8834,8834,True
5,jayggg_ngstents,454,454,True
6,inqwire_sqir,1338,1338,True
7,tee-lab_pyddsde,233,233,True
8,jgcri_gcam-core,10688,0,False
9,windweller_disextract,559,559,True


Submitted complete projects: 7 / 10
Other partially retrieved projects are excluded from this CSV snapshot.


## Required project statistics

The text cells below report the retrieved GitHub metadata and WoC commit counts. Author counts and date ranges are reported only for fully retrieved projects. Partial samples must not be interpreted as full-project statistics.

In [4]:
woc_stats = summary.groupby('project_wocid').agg(ncommits=('commit_sha1','nunique'), nauthors=('author','nunique'), min_time=('date','min'), max_time=('date','max'))
display(woc_stats)

,ncommits,nauthors,min_time,max_time
project_wocid,,,,
avehtari_bda_course_aalto,2272,67,2018-09-07 14:11:11+00:00,2025-11-03 15:40:44+00:00
eberharf_cfl,1138,14,2020-07-17 21:53:13+00:00,2025-05-07 22:32:37+00:00
inqwire_sqir,1338,34,2018-12-20 22:11:21+00:00,2025-07-30 02:27:23+00:00
jayggg_ngstents,454,11,2020-06-29 20:41:15+00:00,2025-06-10 12:50:41+00:00
tee-lab_pyddsde,233,7,2020-03-30 14:40:44+00:00,2021-10-29 06:08:17+00:00
vatlab_sos,8834,55,2016-02-13 00:46:46+00:00,2025-11-06 14:29:35+00:00
windweller_disextract,559,9,2017-10-29 23:56:48+00:00,2025-07-13 15:34:42+00:00


### avehtari_bda_course_aalto

[GitHub repository](https://github.com/avehtari/BDA_course_Aalto)

- GitHub stars: **2282**; forks: **566** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2026-09-29T11:32:50Z**.
- WoC commit count: **2,272** distinct hashes.
- WoC author count: **67 distinct author strings**; minimum time: **2018-09-07T14:11:11+00:00**; maximum time: **2025-11-03T15:40:44+00:00**. Complete SHA coverage verified.

### eberharf_cfl

[GitHub repository](https://github.com/eberharf/cfl)

- GitHub stars: **19**; forks: **1** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2026-05-05T21:52:43Z**.
- WoC commit count: **1,138** distinct hashes.
- WoC author count: **14 distinct author strings**; minimum time: **2020-07-17T21:53:13+00:00**; maximum time: **2025-05-07T22:32:37+00:00**. Complete SHA coverage verified.

### openda-association_openda

[GitHub repository](https://github.com/OpenDA-Association/OpenDA)

- GitHub stars: **104**; forks: **34** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2026-07-10T13:57:20Z**.
- WoC commit count: **3,209** distinct hashes.
- WoC author count, minimum time, maximum time: **unavailable until commit-detail retrieval completes**.

### openms_openms

[GitHub repository](https://github.com/OpenMS/OpenMS)

- GitHub stars: **623**; forks: **445** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2026-09-30T17:00:08Z**.
- WoC commit count: **54,864** distinct hashes.
- WoC author count, minimum time, maximum time: **unavailable until commit-detail retrieval completes**.

### vatlab_sos

[GitHub repository](https://github.com/vatlab/SoS)

- GitHub stars: **286**; forks: **44** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2026-09-09T18:36:17Z**.
- WoC commit count: **8,834** distinct hashes.
- WoC author count: **55 distinct author strings**; minimum time: **2016-02-13T00:46:46+00:00**; maximum time: **2025-11-06T14:29:35+00:00**. Complete SHA coverage verified.

### jayggg_ngstents

[GitHub repository](https://github.com/jayggg/ngstents)

- GitHub stars: **9**; forks: **6** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2025-11-27T00:17:25Z**.
- WoC commit count: **454** distinct hashes.
- WoC author count: **11 distinct author strings**; minimum time: **2020-06-29T20:41:15+00:00**; maximum time: **2025-06-10T12:50:41+00:00**. Complete SHA coverage verified.

### inqwire_sqir

[GitHub repository](https://github.com/inQWIRE/SQIR)

- GitHub stars: **105**; forks: **26** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2025-07-30T02:27:23Z**.
- WoC commit count: **1,338** distinct hashes.
- WoC author count: **34 distinct author strings**; minimum time: **2018-12-20T22:11:21+00:00**; maximum time: **2025-07-30T02:27:23+00:00**. Complete SHA coverage verified.

### tee-lab_pyddsde

[GitHub repository](https://github.com/tee-lab/pyddsde)

- GitHub stars: **98**; forks: **12** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2026-06-10T04:11:37Z**.
- WoC commit count: **233** distinct hashes.
- WoC author count: **7 distinct author strings**; minimum time: **2020-03-30T14:40:44+00:00**; maximum time: **2021-10-29T06:08:17+00:00**. Complete SHA coverage verified.

### jgcri_gcam-core

[GitHub repository](https://github.com/JGCRI/gcam-core)

- GitHub stars: **432**; forks: **226** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2026-06-01T21:04:27Z**.
- WoC commit count: **10,688** distinct hashes.
- WoC author count, minimum time, maximum time: **unavailable until commit-detail retrieval completes**.

### windweller_disextract

[GitHub repository](https://github.com/windweller/DisExtract)

- GitHub stars: **0**; forks: **0** (retrieved 2026-10-08).
- Latest GitHub commit on/before the cutoff: **2025-07-13T15:34:42Z**.
- WoC commit count: **559** distinct hashes.
- WoC author count: **9 distinct author strings**; minimum time: **2017-10-29T23:56:48+00:00**; maximum time: **2025-07-13T15:34:42+00:00**. Complete SHA coverage verified.

## Validation and limitations

The SHA manifest covers all ten assigned projects. Complete-project checks compare exact SHA sets, duplicate keys, author fields, SHA formats, and timestamp parsing. `bdowlin2_source_comparison.csv` reports WoC counts alongside GitHub default-branch counts as of the cutoff. Differences are retained rather than forcing equality across different history scopes. In particular, DisExtract exposes only three current default-branch commits compared with 559 WoC-associated commits.

The README describes an older WoC snapshot, but the retrieved data includes some 2025 commits beyond January. Use each project's actual returned range; no uniform live snapshot date has been verified. Missing commit details are never treated as zero activity.